This cell ensures that the necessary `pub-dialogue` package is installed, either from a local environment or directly from GitHub if running in a Colab environment. This step is crucial for enabling subsequent analyses of the extracted text data, as the package likely contains functions and tools specifically designed for processing and evaluating the quality of public dialogue text related to science and technology.

In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

This cell sets up the necessary libraries and configurations for analyzing paragraph-level text extracted from public dialogue PDFs on science and technology. By defining parameters such as random seeds for reproducibility, model specifications for embeddings and clustering, and processing settings for text chunking, it establishes a foundation for subsequent data quality assessments and ensures that the analysis can handle variations in document structure effectively. This initial setup is crucial for maintaining consistency and accuracy throughout the analysis process, particularly when dealing with diverse and potentially unstructured text data.

In [ ]:
# @title Import libraries and define configuration
import os
import random
import warnings
from pathlib import Path

import numpy as np

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths
PDF_FOLDER = Path("/content/pdfs")
OUTPUT_FOLDER = Path("/content/outputs")
CHECKPOINT_FOLDER = Path("/content/checkpoints")

for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)

warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")


This cell initializes the necessary modules and utility functions from the `pub_dialogue` library, which are essential for assessing the quality of the extracted text from public dialogue PDFs. By importing various functions for data validation, diagnostics, and text processing, it sets the groundwork for subsequent analyses, ensuring that the data can be effectively evaluated and any issues can be addressed systematically. This step is crucial for maintaining the integrity of the analysis, as it directly impacts the reliability of insights drawn from the dialogue data.

In [ ]:
# @title Load pub_dialogue — shared helper functions and access/assess/address modules
try:
    import pub_dialogue.utils as du
    import pub_dialogue.access as access
    import pub_dialogue.assess as assess
    import pub_dialogue.address as address
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("pub_dialogue imported successfully")
except ImportError as _e:
    print(f"pub_dialogue not found: {_e}")
    raise

This cell establishes the necessary API access for interacting with OpenAI's services, which is crucial for processing and analyzing the extracted text from the public dialogue documents. By implementing multiple methods to retrieve the API key, including Colab secrets, environment variables, and manual input, it ensures flexibility and security in different execution environments, thereby facilitating seamless integration of NLP capabilities into the data quality assessment workflow.

In [ ]:
from openai import OpenAI


In [ ]:
# @title Configure API access
import os as _os

api_key = None

# 1. Try Colab secrets (when running in Google Colab)
try:
    from google.colab import userdata as _userdata
    api_key = _userdata.get('OPENAI_API_KEY')
    if api_key:
        print("API key loaded from Colab secrets")
except Exception:
    pass

# 2. Load .env file if present (local development — pip install python-dotenv)
try:
    from dotenv import load_dotenv as _load_dotenv
    _load_dotenv()  # populates os.environ from .env; safe no-op if file absent
except ImportError:
    pass

# 3. Try environment variable (set directly or loaded from .env above)
if not api_key:
    api_key = _os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("API key loaded from environment / .env")

# 4. Interactive prompt fallback
if not api_key:
    from getpass import getpass as _getpass
    api_key = _getpass("Enter OpenAI API key: ")
    print("API key entered manually")

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API connection verified")
except Exception as e:
    print(f"API error: {e}")

This cell generates a visual summary of the data quality for the paragraph-level text extracted from the 65 UK public dialogue PDF documents. By assessing and plotting the quality metrics, it helps identify potential issues in the text data, such as inconsistencies or gaps, which is crucial for ensuring the reliability of subsequent analyses in understanding public perceptions of science and technology.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150


In [ ]:
# @title Summarise paragraph-level data quality
assess.plot_data_quality(chunks_df, OUTPUT_FOLDER)
plt.show()

This cell evaluates the quality of text chunks extracted from the public dialogue documents by identifying those that may be incorrectly classified as meaningful content, such as bibliography entries or survey tables. By reporting the contamination rate based on specified thresholds for word and character counts, it provides critical insights into the reliability of the data, which is essential for ensuring the integrity of subsequent analyses in the study of public dialogue on science and technology.

In [ ]:
# @title (v16) Chunk content-quality diagnostic
# Reports the proportion of kept chunks that look like bibliography fragments
# or survey-table rows, so the analyst can see the contamination rate at the
# chosen MIN_CHUNK_WORDS floor.

show_status("Running chunk content-quality diagnostic...")
chunks_df = assess.flag_chunk_quality(chunks_df, OUTPUT_FOLDER, MIN_CHUNK_WORDS, MIN_CHUNK_CHARS)
show_complete("Chunk quality diagnostic complete — see chunk_quality_flagged.csv")